In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import itertools

import os
'''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
'''
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

2026-05-06 14:33:09.398947: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 14:33:09.410879: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-06 14:33:09.451520: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-06 14:33:09.451550: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-06 14:33:09.451582: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

"\nfor dirname, _, filenames in os.walk('/kaggle/input'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n"

In [9]:
df= pd.read_csv("processed_news_scores_with_stocks_WITH_AGGREGATE_ready_formodeling.csv")
#score_cols = [f"scores_{i}" for i in range(23)]
#df = df.drop(columns=score_cols)

In [10]:
df = df.sort_values(by="timestamp").reset_index(drop=True)


In [11]:
df = df.rename(columns={
    "AAPL": "close_Apple",
    "AMZN": "close_Amazon",
    "TSLA": "close_Tesla"
})

In [12]:
df.columns

Index(['timestamp', 'close_Apple', 'close_Amazon', 'close_Tesla',
       'avg_norm_avg_scores_Tesla', 'avg_norm_max_scores_Tesla',
       'avg_norm_min_scores_Tesla', 'avg_norm_avg_scores_Apple',
       'avg_norm_max_scores_Apple', 'avg_norm_min_scores_Apple',
       'avg_norm_avg_scores_Amazon', 'avg_norm_max_scores_Amazon',
       'avg_norm_min_scores_Amazon'],
      dtype='object')

In [20]:
df.isna().any()

timestamp                     False
close_Apple                    True
close_Amazon                   True
close_Tesla                    True
avg_norm_avg_scores_Tesla     False
avg_norm_max_scores_Tesla     False
avg_norm_min_scores_Tesla     False
avg_norm_avg_scores_Apple     False
avg_norm_max_scores_Apple     False
avg_norm_min_scores_Apple     False
avg_norm_avg_scores_Amazon    False
avg_norm_max_scores_Amazon    False
avg_norm_min_scores_Amazon    False
dtype: bool

# LSTM

## Close price with avg min, and max score of all. 

In [13]:
# convert an array of values into a dataset matrix
def create_dataset(dataset, look_back=1, target_col=0):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), :]          # full feature window
        dataX.append(a)
        dataY.append(dataset[i + look_back, target_col])
    return np.array(dataX), np.array(dataY)




#### Changes implemented on nov 30 code:
1- Adding stopping epochs. 

2- add validation dataset

3- SPLIT then normalize (it intentionally uses train's min/max for everything, including test. Here's why:
In real life, when you deploy a model, you don't know the future's min/max. So you scale future data using what you learned from the past (train). It's the only honest approach.
If test had different min/max than train, that's fine — the scaler still applies train's min/max to test. The test values might go slightly outside [0,1] range, but that's acceptable and actually realistic.)

In [14]:
#######################NEW###########################
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,        # stops if no improvement for 20 epochs
    restore_best_weights=True
)
#######################NEW###########################


In [19]:
n_repeats = 10
lookbacks = np.arange(15, 150, 15)
train_sizes = [0.5, 0.7]
rows = []
best_mse = float("inf")
worst_mse = -float("inf")

# NEW: define companies and their columns
companies = {
    "Tesla":  ["close_Tesla",  "avg_norm_avg_scores_Tesla", "avg_norm_max_scores_Tesla", "avg_norm_min_scores_Tesla"],
    "Apple":  ["close_Apple",  "avg_norm_avg_scores_Apple", "avg_norm_max_scores_Apple", "avg_norm_min_scores_Apple"],
    "Amazon": ["close_Amazon", "avg_norm_avg_scores_Amazon", "avg_norm_max_scores_Amazon", "avg_norm_min_scores_Amazon"],
}

# NEW: outer loop over companies
for company, cols in companies.items():
    best_mse = float("inf")   # NEW: reset per company
    worst_mse = -float("inf") # NEW: reset per company

    for lookback, ts in itertools.product(lookbacks, train_sizes):
        for repeat in range(n_repeats):

            np.random.seed(5)
            dataset = df[cols].values  # NEW: use company-specific columns
            target_col = 0
            
            ######## OLD #####################
            #scaler = MinMaxScaler(feature_range=(0, 1))
            #dataset = scaler.fit_transform(dataset)

            #train_size = int(len(dataset) * ts)
            #train, test = dataset[0:train_size, :], dataset[train_size:, :]
            ######## OLD #####################
            ######## NEW #####################
            
            train_size = int(len(dataset) * ts)
            train, test = dataset[0:train_size, :], dataset[train_size:, :]

            scaler = MinMaxScaler(feature_range=(0, 1))
            train = scaler.fit_transform(train)
            test = scaler.transform(test)
            dataset = np.concatenate([train, test], axis=0)  # needed for plotting later

            
            ######## NEW #####################
            

            look_back = lookback
            trainX, trainY = create_dataset(train, look_back, target_col)
            testX, testY = create_dataset(test, look_back, target_col)

            n_features = dataset.shape[1]
            trainX = trainX.reshape(trainX.shape[0], look_back, n_features)
            testX = testX.reshape(testX.shape[0], look_back, n_features)
            
            print("trainX NaNs:", np.isnan(trainX).sum())
            print("testX NaNs:", np.isnan(testX).sum())
            print("trainY NaNs:", np.isnan(trainY).sum())
            print("testY NaNs:", np.isnan(testY).sum())

            model = Sequential()
            model.add(LSTM(25, input_shape=(look_back, n_features)))
            model.add(Dropout(0.1))
            model.add(Dense(1))
            model.compile(loss='mse', optimizer='adam')
            #model.fit(trainX, trainY, epochs=1000, batch_size=240, verbose=0)
            ##################################NEW################################
            model.fit(trainX, trainY,
              epochs=1000,
              batch_size=240,
              verbose=0,
              validation_split=0.1,   # uses last 10% of trainX as validation
              callbacks=[early_stop])
            
            ##################################NEW################################
            trainPredict = model.predict(trainX)
            testPredict = model.predict(testX)
            
            print("raw trainPredict NaNs:", np.isnan(trainPredict).sum())
            print("raw testPredict NaNs:", np.isnan(testPredict).sum())

            trainPredict = scaler.inverse_transform(np.repeat(trainPredict, n_features, axis=1))[:,target_col]
            trainY_inv = scaler.inverse_transform(np.repeat(trainY.reshape(-1,1), n_features, axis=1))[:,target_col]
            testPredict = scaler.inverse_transform(np.repeat(testPredict, n_features, axis=1))[:,target_col]
            testY_inv = scaler.inverse_transform(np.repeat(testY.reshape(-1,1), n_features, axis=1))[:,target_col]
            
            
            print("trainPredict NaNs:", np.isnan(trainPredict).sum())
            print("trainY_inv NaNs:", np.isnan(trainY_inv).sum())
            print("testPredict NaNs:", np.isnan(testPredict).sum())
            print("testY_inv NaNs:", np.isnan(testY_inv).sum())

            trainScore = math.sqrt(mean_squared_error(trainY_inv, trainPredict))
            testScore = math.sqrt(mean_squared_error(testY_inv, testPredict))

            dataset_target = scaler.inverse_transform(dataset)[:, target_col]

            trainPredictPlot = np.empty_like(dataset_target)
            trainPredictPlot[:] = np.nan
            trainPredictPlot[look_back:len(trainPredict)+look_back] = trainPredict

            testPredictPlot = np.empty_like(dataset_target)
            testPredictPlot[:] = np.nan
            testPredictPlot[len(trainPredict)+(look_back*2)+1 : len(dataset_target)-1] = testPredict

            plt.plot(dataset_target)
            plt.plot(trainPredictPlot)
            plt.plot(testPredictPlot)

            testPrices = dataset_target[len(dataset_target)-len(testPredict):]
            
            

            trainScore = math.sqrt(mean_squared_error(trainY_inv, trainPredict))
            testScore = math.sqrt(mean_squared_error(testY_inv, testPredict))

            if testScore < best_mse:
                best_mse = testScore
                plt.figure(figsize=(12,6))
                plt.plot(dataset_target, label="Actual Prices (Full Dataset)", color="blue")
                plt.plot(trainPredictPlot, label="Train Predictions", color="orange")
                plt.plot(testPredictPlot, label="Test Predictions", color="green")
                plt.legend()
                plt.title(f"LSTM Predictions vs Actual Prices - {company}") # NEW: company in title
                plt.xlabel("Time Steps")
                plt.ylabel("Price")
                plt.grid(True)
                plt.savefig(f"LSTM_Model2_Regression_best_mse_{company}_lb{lookback}_ts{ts}_repeat{repeat}.png") # NEW: company in filename
                testPredictPlot=pd.Series(testPredictPlot).dropna().values
                test_size = testPredictPlot.shape[0]
                plt.figure(figsize=(12,6))
                plt.plot(dataset_target[-test_size:], label="Actual Prices (Full Dataset)", color="blue")
                plt.plot(testPredictPlot, label="Test Predictions", color="green")
                plt.legend()
                plt.title(f"LSTM Predictions vs Actual Prices - {company}") # NEW: company in title
                plt.xlabel("Time Steps")
                plt.ylabel("Price")
                plt.grid(True)

            if testScore > worst_mse:
                worst_mse = testScore
                plt.figure(figsize=(12,6))
                plt.plot(dataset_target, label="Actual Prices (Full Dataset)", color="blue")
                plt.plot(trainPredictPlot, label="Train Predictions", color="orange")
                plt.plot(testPredictPlot, label="Test Predictions", color="green")
                plt.legend()
                plt.title(f"LSTM Predictions vs Actual Prices - {company}") # NEW: company in title
                plt.xlabel("Time Steps")
                plt.ylabel("Price")
                plt.grid(True)
                plt.savefig(f"LSTM_Model2_Regression_worst_mse_{company}_lb{lookback}_ts{ts}_repeat{repeat}.png") # NEW: company in filename
                testPredictPlot=pd.Series(testPredictPlot).dropna().values
                test_size = testPredictPlot.shape[0]
                plt.figure(figsize=(12,6))
                plt.plot(dataset_target[-test_size:], label="Actual Prices (Full Dataset)", color="blue")
                plt.plot(testPredictPlot, label="Test Predictions", color="green")
                plt.legend()
                plt.title(f"LSTM Predictions vs Actual Prices - {company}") # NEW: company in title
                plt.xlabel("Time Steps")
                plt.ylabel("Price")
                plt.grid(True)

            result = {"company": company, "n_repeat": repeat, "lookback": look_back, "test_size": 1-ts, "train_rmse": trainScore, "test_rmse": testScore} # NEW: company column
            rows.append(result)

df_results = pd.DataFrame(rows)

trainX NaNs: 1740
testX NaNs: 45
trainY NaNs: 116
testY NaNs: 3
368/368 [==============================] - 1s 1ms/step
raw trainPredict NaNs: 11749
raw testPredict NaNs: 11750
trainPredict NaNs: 11749
trainY_inv NaNs: 116
testPredict NaNs: 11750
testY_inv NaNs: 3


ValueError: Input contains NaN.

In [17]:

df_results.to_csv("nov30_codes_new_data/LSTM_LLM_Regression_improved_simulation_NEW_DATA.csv", index=False)